# Qwen-Mythos: Induced Recurrence on Qwen3.6-35B-A3B

**Hypothesis**: Qwen3.6-35B-A3B is a standard architecture (no latent loop). Our 5-level null intervention results are consistent with the absence of latent planning. If we **induce artificial recurrence** over layers L11→L22 (where Cohen's d peaks in our S4-L0 mapping), we simulate Saunshi et al. 2025's "implicit CoT in latent space".

Four testable outcomes:

| Result | Interpretation |
|---|---|
| More loops → higher accuracy | Latent reasoning exists, just not triggered by standard forward — **proof-of-concept for looped inference** |
| More loops → same accuracy | Retrieval-dominated, no iteration benefit — **paper reframes as binding circuit** |
| More loops → lower accuracy | Instability; standard weights don't tolerate recurrence — **motivates Parcae-style training** |
| Moderate loops help, many hurt | Optimal recurrence depth exists — **scaling law within single model** |

**Pipeline** (~2h on RTX 6000 Blackwell 96GB):
1. Install + load model (~10 min)
2. Implement `LoopHookManager` — captures kwargs at L11, intercepts L23, replays L11-L22 N times (~5 min)
3. Benchmark: 50 SuperGPQA questions, N_loops ∈ {0, 1, 2, 4}, measure answer letter logit (~90 min)
4. Analysis + verdict

**Positions chosen**:
- L11 = `LOOP_START` — start of Gated-Attn layers in our architecture map (Gated-Attn at {3, 7, 11, 15, 19, 23, 27, 31, 35, 39})
- L22 = `LOOP_END` — 1 before our Cohen's d peak L22, 17 before logit-lens commit L39
- L23 = intercept point (adjacent to peak, still dense-enough for hooks)

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping colab auth: {e})')

import json, re, pickle, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/mythos_loop')
OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load model + identify layer structure

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Find layers
def get_layers(m):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                return cur, path
    raise RuntimeError('layers not found')

all_layers, layer_path = get_layers(model)
print(f'Found {len(all_layers)} layers at path {layer_path}')
print(f'Layer 11 type: {type(all_layers[11]).__name__}')
print(f'Layer 22 type: {type(all_layers[22]).__name__}')
print(f'Layer 23 type: {type(all_layers[23]).__name__}')

## Cell 3 — LoopHookManager

Approach: capture forward-call kwargs at L11 (start of loop block). Intercept L23's pre-input — replay layers L11..L22 N extra times using captured kwargs. Return modified hidden_state to L23.

**Caveat**: this works for a SINGLE forward pass (not autoregressive generation with KV cache, since cache invalidates after replay). We use greedy with full recomputation.

In [ ]:
LOOP_START = 11
LOOP_END = 22
INTERCEPT = 23

class LoopHookManager:
    def __init__(self, layers, start, end, intercept):
        self.layers = layers
        self.start = start
        self.end = end
        self.intercept = intercept
        self.n_loops = 0
        self.captured_kwargs = None
        self.h_capture = None
        self.h_intercept = None
        self.applied = False

    def set_n_loops(self, n):
        self.n_loops = n
        self.applied = False
        self.captured_kwargs = None

    def _capture_hook(self, module, args, kwargs):
        # Save the kwargs that layer[start] was called with (attention_mask, position_ids, etc)
        if self.captured_kwargs is None:
            self.captured_kwargs = {k: v for k, v in kwargs.items() if k != 'past_key_values'}
        return None

    def _intercept_hook(self, module, args, kwargs):
        if self.n_loops == 0 or self.applied or self.captured_kwargs is None:
            return None
        # args[0] is hidden_states entering layer[intercept]
        hidden = args[0] if len(args) > 0 else kwargs.get('hidden_states')
        if hidden is None:
            return None
        # Replay L_start..L_end for n_loops extra iterations
        safe_kwargs = {k: v for k, v in self.captured_kwargs.items() if k != 'past_key_values'}
        for _ in range(self.n_loops):
            for i in range(self.start, self.end + 1):
                out = self.layers[i](hidden, **safe_kwargs)
                hidden = out[0] if isinstance(out, tuple) else out
        self.applied = True
        if len(args) > 0:
            new_args = (hidden,) + args[1:]
            return new_args, kwargs
        else:
            kwargs['hidden_states'] = hidden
            return args, kwargs

    def install(self):
        self.h_capture = self.layers[self.start].register_forward_pre_hook(
            self._capture_hook, with_kwargs=True)
        self.h_intercept = self.layers[self.intercept].register_forward_pre_hook(
            self._intercept_hook, with_kwargs=True)

    def uninstall(self):
        if self.h_capture: self.h_capture.remove()
        if self.h_intercept: self.h_intercept.remove()

    def reset(self):
        self.applied = False
        self.captured_kwargs = None

loop_mgr = LoopHookManager(all_layers, LOOP_START, LOOP_END, INTERCEPT)
loop_mgr.install()
print(f'✅ LoopHookManager installed: capture L{LOOP_START}, intercept L{INTERCEPT}, replay L{LOOP_START}..L{LOOP_END}')

## Cell 4 — Smoke test: measure output stability under N_loops

Before full eval: verify that looping doesn't blow up. Run a toy prompt under N_loops ∈ {0, 1, 2, 4, 8} and measure:
- Output logit norm at last position
- Next token prediction (should vary or stay stable predictably)
- Residual norm at last layer

In [ ]:
TEST_PROMPT = "The capital of France is"
ids = tok(TEST_PROMPT, return_tensors='pt').input_ids.to('cuda')
print(f'Input ids shape: {ids.shape}')

print('\nN_loops | last-token logit max | next-token | residual norm')
for n in [0, 1, 2, 4, 8]:
    loop_mgr.set_n_loops(n)
    loop_mgr.reset()
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=torch.ones_like(ids), output_hidden_states=True)
    logits = out.logits[0, -1]  # last position
    next_id = logits.argmax().item()
    next_tok = tok.decode([next_id])
    resid = out.hidden_states[-1][0, -1]
    print(f'{n:7d} | {logits.max().item():>21.3f} | {next_tok!r:>14s} | {resid.norm().item():>12.3f}')

loop_mgr.set_n_loops(0)
print('\n✅ smoke test done')

## Cell 5 — Load SuperGPQA test set + prompt formatter

We use 50 questions from Stage B. Protocol: encode `prompt + options + "The answer is \\boxed{"` and measure the probability of each letter token at the NEXT position. No generation — direct logit readout at the answer position.

In [ ]:
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
for shard in tqdm(shards, desc='load questions'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q_options = json.loads(meta['options'])
        rollouts = json.loads(meta['rollouts'])
        n_correct = sum(1 for r in rollouts if r['correct'])
        if n_correct >= 2:  # medium-easy filter for cleaner baseline
            questions.append(dict(
                question=meta['question'], options=q_options,
                gold_letter=meta['gold_letter'],
                n_options=len(q_options),
                n_correct_rollouts=n_correct,
            ))
print(f'{len(questions)} questions with ≥2 correct rollouts')

def format_mcq_for_direct(question, options):
    """Format for direct letter logit readout. Ends at \\boxed{ so next token should be letter."""
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = (
        "Answer the following multiple-choice question. "
        "Give ONLY the letter of the correct answer.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}\n\nAnswer: \\boxed{{")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

# Cache letter token IDs (single token per letter — filter to single-letter tokens only)
letter_ids = {}
for L in 'ABCDEFGHIJ':
    ids_variants = []
    for s in [L, f' {L}']:
        tok_ids = tok(s, add_special_tokens=False).input_ids
        if len(tok_ids) == 1:
            ids_variants.append(tok_ids[0])
    # Take the first single-token ID (usually ' A')
    letter_ids[L] = ids_variants[0] if ids_variants else tok(L, add_special_tokens=False).input_ids[0]
print('letter → token_id:', letter_ids)

# Smoke test on one question
q0 = questions[0]
p0 = format_mcq_for_direct(q0['question'], q0['options'])
print(f'\nSample prompt (last 500 chars):\n{p0[-500:]}')

## Cell 6 — Eval: N_loops × N_questions

For each question, compute P(letter) = softmax(logits)[letter_id] for each letter at the final position.
Measure accuracy as `argmax(P(letter)) == gold_letter`.

Sweep N_loops ∈ {0, 1, 2, 4}.

In [ ]:
N_TEST = 50
LOOP_CONFIGS = [0, 1, 2, 4]

random.seed(42)
sample = random.sample(questions, min(N_TEST, len(questions)))
print(f'Evaluating {len(sample)} questions × {len(LOOP_CONFIGS)} loop configs')

results = []
t0 = time.time()
for i, q in enumerate(tqdm(sample, desc='mythos eval')):
    try:
        p = format_mcq_for_direct(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096:
            continue  # skip very long

        row = dict(
            idx=i, gold=q['gold_letter'], n_options=q['n_options'],
            n_correct_rollouts=q['n_correct_rollouts'])

        for n_loops in LOOP_CONFIGS:
            loop_mgr.set_n_loops(n_loops)
            loop_mgr.reset()
            with torch.no_grad():
                out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            logits = out.logits[0, -1]
            # Letter probs
            letter_logits = {L: logits[letter_ids[L]].item() for L in 'ABCDEFGHIJ'[:q['n_options']]}
            pred = max(letter_logits, key=letter_logits.get)
            row[f'loops_{n_loops}_pred'] = pred
            row[f'loops_{n_loops}_correct'] = (pred == q['gold_letter'])
            row[f'loops_{n_loops}_gold_logit'] = letter_logits[q['gold_letter']]
            row[f'loops_{n_loops}_argmax_logit'] = letter_logits[pred]

        results.append(row)
        if (i+1) % 5 == 0:
            for n_loops in LOOP_CONFIGS:
                acc = sum(r[f'loops_{n_loops}_correct'] for r in results) / len(results)
                print(f'  [{i+1}/{N_TEST}] N={n_loops} acc={acc*100:.0f}%', end=' | ')
            print()
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        continue

loop_mgr.set_n_loops(0)
with open(OUT/'mythos_results.json', 'w') as f:
    json.dump(dict(n=len(results), configs=LOOP_CONFIGS, results=results,
                   wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ done in {(time.time()-t0)/60:.1f} min')

## Cell 7 — Analysis + verdict

In [ ]:
print('=== Mythos-Loop results on SuperGPQA (N={}) ==='.format(len(results)))
print(f'{"N_loops":8s} | {"accuracy":>9s} | {"mean_gold_logit":>15s} | {"mean_argmax_logit":>17s}')
accs = []
for n_loops in LOOP_CONFIGS:
    acc = sum(r[f'loops_{n_loops}_correct'] for r in results) / len(results)
    mean_gold = np.mean([r[f'loops_{n_loops}_gold_logit'] for r in results])
    mean_amax = np.mean([r[f'loops_{n_loops}_argmax_logit'] for r in results])
    print(f'{n_loops:>8d} | {acc*100:>8.1f}% | {mean_gold:>15.3f} | {mean_amax:>17.3f}')
    accs.append(acc)

baseline_acc = accs[0]
max_acc = max(accs[1:])
max_n = LOOP_CONFIGS[1 + accs[1:].index(max_acc)]
delta = (max_acc - baseline_acc) * 100

print(f'\nBest looped acc: N={max_n} at {max_acc*100:.1f}% (baseline N=0: {baseline_acc*100:.1f}%)')
print(f'Δ (max looped − baseline) = {delta:+.1f}pp')

# Verdict
print('\n=== VERDICT ===')
if delta > 5:
    print(f'✅ LATENT REASONING EXISTS — induced recurrence improves by {delta:+.1f}pp')
    print('   → paper: "Inducing artificial recurrence in hybrid MoE improves reasoning (training-free)"')
    print('   → publishable as proof-of-concept + scales to larger N + more benchmarks')
elif -5 < delta < 5:
    print(f'⚖️  NO SIGNIFICANT EFFECT ({delta:+.1f}pp) — retrieval-dominated or non-iterable reasoning')
    print('   → paper: "Hybrid MoE architectures do not benefit from induced recurrence — retrieval signature"')
    print('   → null is informative; negative result well-packaged')
else:
    print(f'❌ LOOP DESTABILIZES ({delta:+.1f}pp) — weights not compatible with recurrence')
    print('   → paper: "Standard Qwen3.6 weights are not trained for looping; motivates Parcae-style training"')
    print('   → motivates retraining experiments (Paper 2)')

# Also report delta per loops
print('\n=== Per-N delta vs baseline ===')
for n_loops, acc in zip(LOOP_CONFIGS, accs):
    if n_loops == 0: continue
    d = (acc - baseline_acc) * 100
    marker = '⭐' if d > 5 else ('✗' if d < -5 else '≈')
    print(f'  N={n_loops}: {acc*100:.1f}% (Δ = {d:+.1f}pp) {marker}')